# Stress Prediction V39 — 잔차(오차) 분석

지금까지 시도한 피처/모델 추가는 전부 v34(CV 0.148 / Public 0.127)를 못 넘었습니다.
방향을 바꿔서 **v34 OOF 예측의 오차가 어느 그룹에서 특히 큰지**를 먼저 확인합니다.
"무작정 추가"가 아니라 데이터가 알려주는 약점을 근거로 다음 시도를 정하기 위한 진단 노트북입니다.

1. v34 원본 로직(tree quantile trick + pair, 0.72/0.28 블렌드)으로 5-Fold OOF 예측 생성
2. 범주형 컬럼별(성별/흡연/수면패턴/학력/활동/병력 등) 평균 절대오차(MAE) 비교
3. 연속형 컬럼별(나이/BMI/혈압 등) 5구간으로 나눠 구간별 MAE 비교
4. 오차가 가장 큰 상위 N개 샘플 직접 확인


In [ ]:

from __future__ import annotations

import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.54
PAIR_QUANTILE = 0.48
PAIR_WEIGHT = 0.28

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def tree_prediction(train_features, val_features, target) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_val = add_feature_copies(val_features)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS, min_samples_leaf=1, max_features=1,
        criterion="squared_error", random_state=RANDOM_SEED, n_jobs=-1,
    )
    model.fit(train_matrix, target)
    tree_predictions = np.column_stack([t.predict(val_matrix) for t in model.estimators_])
    return np.quantile(tree_predictions, TREE_QUANTILE, axis=1)


def empirical_rank_transform(train_values, test_values):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    test_rank = np.searchsorted(sorted_train, test_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_prediction(train_features, val_features, target) -> np.ndarray:
    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )
    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_idx = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_idx])
    return np.quantile(np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1)


## 1. v34 OOF 예측 생성

In [ ]:

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
train = pd.read_csv(DATA_DIR / "train.csv")

raw_X = add_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)
oof_pred = np.zeros(len(y))

for fold, (train_idx, val_idx) in enumerate(cv.split(raw_X), start=1):
    train_feat = raw_X.iloc[train_idx].reset_index(drop=True)
    val_feat = raw_X.iloc[val_idx].reset_index(drop=True)
    y_train_fold = y.iloc[train_idx].reset_index(drop=True)

    pred_tree = tree_prediction(train_feat, val_feat, y_train_fold)
    pred_pair = pair_prediction(train_feat, val_feat, y_train_fold)
    combined = np.clip(np.round((1 - PAIR_WEIGHT) * pred_tree + PAIR_WEIGHT * pred_pair, 2), 0.0, 1.0)
    oof_pred[val_idx] = combined
    print(f"Fold {fold} done")

print("전체 OOF MAE:", f"{mean_absolute_error(y, oof_pred):.6f}")

analysis = raw_X.copy()
analysis["y_true"] = y
analysis["y_pred"] = oof_pred
analysis["abs_error"] = np.abs(analysis["y_true"] - analysis["y_pred"])
analysis["signed_error"] = analysis["y_pred"] - analysis["y_true"]  # 양수 = 과대예측


## 2. 범주형 컬럼별 오차 비교

전체 평균(`전체 OOF MAE`)보다 확실히 높은 그룹이 있는지 확인하세요.
그룹 크기(`count`)가 너무 작으면(예: 10개 미만) 우연일 수 있으니 같이 봐야 합니다.


In [ ]:

categorical_cols = raw_X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
print("범주형 컬럼:", categorical_cols)
print()

for col in categorical_cols:
    summary = analysis.groupby(col)["abs_error"].agg(["mean", "count", "std"]).sort_values("mean", ascending=False)
    print(f"### {col}")
    display(summary)
    print()


## 3. 연속형 컬럼별 오차 비교 (5구간 분위수 분할)

In [ ]:

numeric_cols = raw_X.select_dtypes(include=["number"]).columns.tolist()

for col in numeric_cols:
    try:
        analysis[f"{col}_bin"] = pd.qcut(raw_X[col], q=5, duplicates="drop")
    except ValueError:
        continue  # 값 종류가 너무 적어서 5구간으로 못 나누는 컬럼은 스킵
    summary = analysis.groupby(f"{col}_bin", observed=True)["abs_error"].agg(["mean", "count"])
    print(f"### {col}")
    display(summary)
    print()


## 4. 오차가 가장 큰 상위 30개 샘플

여기서 특정 그룹(성별/흡연/수면패턴 조합 등)이 몰려 있는지 눈으로 확인해보세요.


In [ ]:

worst_cases = analysis.sort_values("abs_error", ascending=False).head(30)
display_cols = ["y_true", "y_pred", "abs_error", "signed_error"] + categorical_cols + ["age"]
display(worst_cases[[c for c in display_cols if c in worst_cases.columns]])
